# Learning Objectives

In this notebook, you will craft sophisticated ETL jobs that interface with a variety of common data sources, such as 
- REST APIs (HTTP endpoints)
- RDBMS
- Hive tables (managed tables)
- Various file formats (csv, json, parquet, etc.)

d

# Interview Questions

As you progress through the practice, attempt to answer the following questions:

## Columnar File
- What is a columnar file format and what advantages does it offer?
- Why is Parquet frequently used with Spark and how does it function?
- How do you read/write data from/to a Parquet file using a DataFrame?

## Partitions
- How do you save data to a file system by partitions? (Hint: Provide the code)
- How and why can partitions reduce query execution time? (Hint: Give an example)

## JDBC and RDBMS
- How do you load data from an RDBMS into Spark? (Hint: Discuss the steps and JDBC)

## REST API and HTTP Requests
- How can Spark be used to fetch data from a REST API? (Hint: Discuss making API requests)

## ETL Job One: Parquet file
### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Data transformation requirements https://pgexercises.com/questions/aggregates/fachoursbymonth.html

### Load
Load data into a parquet file

### What is Parquet? 

Columnar files are an important technique for optimizing Spark queries. Additionally, they are often tested in interviews.
- https://www.youtube.com/watch?v=KLFadWdomyI
- https://www.databricks.com/glossary/what-is-parquet

In [ ]:
from pyspark.sql.functions import col, sum as _sum

# Extract
bookings = spark.table("bookings_csv")
members = spark.table("members_csv")
facilities = spark.table("facilities_csv")

# Transform: total slots booked per facility in September 2012, sorted by slots
# https://pgexercises.com/questions/aggregates/fachoursbymonth.html
fachoursbymonth_df = (
    bookings
    .filter((col("starttime") >= "2012-09-01") & (col("starttime") < "2012-10-01"))
    .groupBy("facid")
    .agg(_sum("slots").alias("Total Slots"))
    .orderBy("Total Slots")
)

display(fachoursbymonth_df)

# Load: write result to a parquet file
parquet_path = "/FileStore/output/fachoursbymonth.parquet"
fachoursbymonth_df.write.mode("overwrite").parquet(parquet_path)

# Sanity check: read it back
display(spark.read.parquet(parquet_path))


## ETL Job Two: Partitions

### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Transform the data https://pgexercises.com/questions/joins/threejoin.html

### Load
Partition the result data by facility column and then save to `threejoin_delta` managed table. Additionally, they are often tested in interviews.

hint: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.partitionBy.html

What are paritions? 

Partitions are an important technique to optimize Spark queries
- https://www.youtube.com/watch?v=hvF7tY2-L3U&t=268s

In [ ]:
from pyspark.sql.functions import col, concat_ws

# Extract
bookings = spark.table("bookings_csv")
members = spark.table("members_csv")
facilities = spark.table("facilities_csv")

# Transform: members who have used a tennis court, with facility name and
# member name as a single column, deduplicated, ordered by member name
# https://pgexercises.com/questions/joins/threejoin.html
threejoin_df = (
    bookings
    .join(members, "memid")
    .join(facilities, "facid")
    .filter(col("name").like("Tennis Court%"))
    .select(
        concat_ws(" ", col("firstname"), col("surname")).alias("member"),
        col("name").alias("facility"),
    )
    .distinct()
    .orderBy("member")
)

display(threejoin_df)

# Load: partition by facility, save as a managed (Delta) table
spark.sql("DROP TABLE IF EXISTS threejoin_delta")

(
    threejoin_df.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("facility")
    .saveAsTable("threejoin_delta")
)

display(spark.sql("SELECT * FROM threejoin_delta"))


## ETL Job Three: HTTP Requests

### Extract
Extract daily stock price data price from the following companies, Google, Apple, Microsoft, and Tesla. 

Data Source
- API: https://rapidapi.com/alphavantage/api/alpha-vantage
- Endpoint: GET `TIME_SERIES_DAILY`

Sample HTTP request

```
curl --request GET \
	--url 'https://alpha-vantage.p.rapidapi.com/query?function=TIME_SERIES_DAILY&symbol=TSLA&outputsize=compact&datatype=json' \
	--header 'X-RapidAPI-Host: alpha-vantage.p.rapidapi.com' \
	--header 'X-RapidAPI-Key: [YOUR_KEY]'

```

Sample Python HTTP request

```
import requests

url = "https://alpha-vantage.p.rapidapi.com/query"

querystring = {
    "function":"TIME_SERIES_DAILY",
    "symbol":"IBM",
    "datatype":"json",
    "outputsize":"compact"
}

headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": "[YOUR_KEY]"
}

response = requests.get(url, headers=headers, params=querystring)

data = response.json()

# Now 'data' contains the daily time series data for "IBM"
```

### Transform
Find **weekly** max closing price for each company.

hints: 
  - Use a `for-loop` to get stock data for each company
  - Use the spark `union` operation to concat all data into one DF
  - create a new `week` column from the data column
  - use `group by` to calcualte max closing price

### Load
- Partition `DF` by company
- Load the DF in to a managed table called, `max_closing_price_weekly`

In [ ]:
import time
import requests
from pyspark.sql import Row
from pyspark.sql.functions import col, date_trunc, max as _max

# NOTE: You need your own RapidAPI key for the Alpha Vantage API.
# Set it as a Databricks secret and read it, rather than hard-coding it, e.g.:
#   api_key = dbutils.secrets.get(scope="alpha-vantage", key="rapidapi-key")
api_key = "[YOUR_KEY]"

url = "https://alpha-vantage.p.rapidapi.com/query"
headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": api_key,
}

# Google (GOOGL), Apple (AAPL), Microsoft (MSFT), Tesla (TSLA)
companies = ["GOOGL", "AAPL", "MSFT", "TSLA"]

dfs = []

# Extract: loop over each company and fetch its daily time series
for symbol in companies:
    querystring = {
        "function": "TIME_SERIES_DAILY",
        "symbol": symbol,
        "datatype": "json",
        "outputsize": "compact",
    }

    response = requests.get(url, headers=headers, params=querystring)
    data = response.json()

    daily_series = data.get("Time Series (Daily)", {})

    rows = [
        Row(
            company=symbol,
            date=date_str,
            close=float(values["4. close"]),
        )
        for date_str, values in daily_series.items()
    ]

    company_df = spark.createDataFrame(rows)
    dfs.append(company_df)

    # Free tier rate limiting: Alpha Vantage allows 5 requests/minute
    time.sleep(15)

# Union all company DataFrames into a single DF
stocks_df = dfs[0]
for df in dfs[1:]:
    stocks_df = stocks_df.union(df)

# Transform: derive a week column, then find the weekly max closing price per company
weekly_df = stocks_df.withColumn("week", date_trunc("week", col("date").cast("date")))

max_closing_price_weekly_df = (
    weekly_df
    .groupBy("company", "week")
    .agg(_max("close").alias("max_close"))
    .orderBy("company", "week")
)

display(max_closing_price_weekly_df)

# Load: partition by company, save as a managed table
spark.sql("DROP TABLE IF EXISTS max_closing_price_weekly")

(
    max_closing_price_weekly_df.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("company")
    .saveAsTable("max_closing_price_weekly")
)

display(spark.sql("SELECT * FROM max_closing_price_weekly"))


## ETL Job Four: RDBMS


### Extract
Extract RNA data from a public PostgreSQL database.

- https://rnacentral.org/help/public-database
- Extract 100 RNA records from the `rna` table (hint: use `limit` in your sql)
- hint: use `spark.read.jdbc` https://docs.databricks.com/external-data/jdbc.html

### Transform
We want to load the data as it so there is no transformation required.


### Load
Load the DF in to a managed table called, `rna_100_records`

In [ ]:
# Public RNAcentral PostgreSQL database connection details
# https://rnacentral.org/help/public-database
jdbc_hostname = "hh-pgsql-public.ebi.ac.uk"
jdbc_port = 5432
jdbc_database = "pfmegrnargs"
jdbc_url = f"jdbc:postgresql://{jdbc_hostname}:{jdbc_port}/{jdbc_database}"

connection_properties = {
    "user": "reader",
    "password": "NWDMCE5xdipIjRrp",
    "driver": "org.postgresql.Driver",
}

# Extract: read 100 records from the `rna` table via JDBC.
# We push the LIMIT down using a subquery passed as the "table" so only
# 100 rows are pulled across the wire, rather than the whole table.
rna_query = "(SELECT * FROM rna LIMIT 100) AS rna_subset"

rna_df = spark.read.jdbc(
    url=jdbc_url,
    table=rna_query,
    properties=connection_properties,
)

display(rna_df)

# Transform: none required - load the data as-is

# Load
spark.sql("DROP TABLE IF EXISTS rna_100_records")
rna_df.write.saveAsTable("rna_100_records")

display(spark.sql("SELECT * FROM rna_100_records"))
